# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar el modelo que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

### ¿Qué hace esta celda?

Esta celda verifica que Google Colab tenga una **GPU NVIDIA** disponible. `nvidia-smi` permite consultar el modelo de GPU y la memoria asignada al entorno.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Verificar que Colab asignó una GPU

# Este comando consulta la GPU disponible en Colab y muestra su nombre y memoria total.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga del modelo.

### Instalación de Ollama

**Ollama** permite descargar y ejecutar modelos de lenguaje dentro de la propia máquina virtual de Colab. Aquí se instala una dependencia de compresión y después el programa Ollama.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

# Instala zstd, una utilidad de compresión utilizada por el entorno.
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
# Descarga y ejecuta el instalador oficial de Ollama.
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

# Confirma en pantalla que la instalación terminó.
print("Ollama instalado.")

Ollama instalado.


### Iniciar el servidor local de Ollama

Ollama funciona mediante un **servidor local**. Esta celda lo inicia en segundo plano y comprueba que responda en `localhost:11434` antes de continuar.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

# Importa herramientas para iniciar procesos, esperar y hacer solicitudes HTTP.
import subprocess
# Inicia `ollama serve` como un proceso en segundo plano.
import time
# Intenta conectarse al servidor hasta 30 veces mientras termina de arrancar.
import requests

# Consulta la versión de Ollama instalada.
servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.3


### **DESCARGAR EL MODELO**

Vamos a trabajar con **Llama 3.1 de 8B**. Guardamos el nombre en la variable `MODELO` para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

### Descargar el modelo Llama

Aquí se define el modelo que se utilizará durante la práctica y se descarga con `ollama pull`. `ollama list` permite comprobar qué modelos están disponibles localmente.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Descargar el modelo Llama (la primera vez tarda unos minutos)

# Guarda el nombre exacto del modelo en una variable para reutilizarlo.
MODELO = "llama3.1:8b"

# Descarga los archivos del modelo Llama en la máquina virtual.
!ollama pull {MODELO}
# Muestra los modelos que Ollama tiene disponibles localmente.
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


### **CLIENTE DE PYTHON**

### Conectar Python con Ollama

La librería `ollama` actúa como **cliente de Python**. El cliente envía solicitudes al servidor local que está ejecutando el modelo.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Instalar el cliente de Ollama y conectarlo al servidor local

# Instala el cliente de Python para Ollama.
!pip install ollama -q

# Importa la librería ollama.
import ollama

# Crea un cliente conectado al servidor local de Ollama.
client = ollama.Client(host="http://localhost:11434")
# Confirma que el cliente quedó inicializado.
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

### Zero-shot

En **zero-shot** se pide una tarea al modelo sin proporcionarle ejemplos previos. Aquí se solicita clasificar el sentimiento de una reseña directamente.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Prompt de clasificación en modo zero-shot

# Define la instrucción y la reseña que el modelo debe clasificar.
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

# Envía el prompt al modelo mediante el cliente de Ollama.
response_zero = client.chat(
# Muestra la clasificación generada por el modelo.
    model=MODELO,
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.message.content)

Zero-shot: Negativo.


### Few-shot

En **few-shot** se incluyen algunos ejemplos de entrada y salida antes de la consulta real. Los ejemplos ayudan al modelo a inferir el formato y las categorías esperadas.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Prompt de clasificación en modo few-shot

# Construye un prompt con ejemplos de reseñas ya clasificadas.
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.


Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

### Razonamiento paso a paso

Esta sección solicita al modelo que desarrolle un problema antes de presentar la respuesta final. El objetivo es observar cómo cambia la salida cuando el prompt exige una explicación estructurada.


In [ ]:
# Razonamiento paso a paso (chain-of-thought)

# Define el problema que se enviará al modelo.
problema = (
# Solicita al modelo una respuesta con desarrollo paso a paso.
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
# Muestra el resultado generado.
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

Analicemos el problema paso a paso:

1. El primer tren sale de la ciudad A a 80 km/h y viaja durante dos horas, por lo que la distancia recorrida por el primer tren es:

Distancia = Velocidad x Tiempo
= 80 km/h x 2 h
= 160 km

2. El segundo tren sale de la ciudad A dos horas después y viaja a 120 km/h. Necesitamos encontrar la distancia recorrida por el segundo tren en ese momento. Como ha salido dos horas después, el primer tren ya habrá recorrido 160 km, por lo que la distancia recorrida por el segundo tren será:

Distancia = Velocidad x Tiempo
= 120 km/h x 2 h
= 240 km

3. Sin embargo, como el segundo tren ha comenzado a viajar dos horas después que el primer tren, ha recorrido 160 km en ese tiempo. Por lo tanto, la distancia que aún le queda al segundo tren para alcanzar al primero es:

Distancia restante = Distancia recorrida por el segundo tren - Distancia recorrida por el primer tren
= 240 km - 160 km
= 80 km

4. Ahora, necesitamos encontrar el tiempo que tardará el segundo tren

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

### Límite del conocimiento y alucinaciones

Aquí se formula una pregunta sobre un evento específico que el modelo no tiene forma garantizada de conocer. La intención es observar que una respuesta fluida o segura **no garantiza que sea verdadera**.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

# Construye una pregunta sobre información específica no proporcionada al modelo.
prompt_desconocido = (
# Envía la pregunta a Llama.
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
# Muestra la respuesta para analizar si el modelo reconoce su límite o inventa información.
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación

Lo siento, pero no tengo información sobre el resultado de la final del hackathon interno de DEV.F del 14 de agosto de 2026, ya que no hay detalles disponibles sobre un evento específico con esa fecha y organización.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

### Preparar los embeddings para RAG

Para recuperar información por significado utilizaremos `sentence-transformers`. Esta biblioteca transforma textos en **vectores numéricos (embeddings)** que después pueden compararse matemáticamente.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Instalar sentence-transformers

# Instala la biblioteca sentence-transformers.
!pip install sentence-transformers --quiet

# Importa el modelo de embeddings y NumPy para operaciones numéricas.
from sentence_transformers import SentenceTransformer
import numpy as np

### Base de conocimiento y embeddings

Se crea una pequeña base de conocimiento con políticas de devolución. Después cada fragmento se convierte en un embedding para poder buscar el contenido más relacionado con una pregunta.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

# Carga un modelo multilingüe que convierte texto en embeddings.
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Define los fragmentos que formarán la base de conocimiento.
documentos = [
# Convierte cada fragmento en un vector numérico.
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


### Recuperación del fragmento más relevante

Esta función convierte la pregunta en un embedding y compara ese vector con los embeddings de los documentos. Luego selecciona el fragmento con la mayor similitud.


In [ ]:
# Comentarios de apoyo: el código ejecutable original se conserva sin cambios.
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

# Define una función para recuperar el fragmento más relacionado con una pregunta.
def buscar_fragmento(pregunta):
# Convierte la pregunta en un embedding.
    embedding_pregunta = modelo_embeddings.encode([pregunta])
# Calcula la similitud entre la pregunta y todos los documentos.
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
# Localiza el índice con la similitud más alta.
    indice_mas_similar = np.argmax(similitudes)
# Devuelve el documento correspondiente.
    return documentos[indice_mas_similar]

# Define una pregunta de prueba, recupera el fragmento y lo muestra.
pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


### Generación aumentada con recuperación (RAG)

En esta última parte, el fragmento recuperado se incorpora al prompt como **contexto externo**. Llama recibe tanto la pregunta como la política relevante y genera una respuesta basada en ese contenido.


In [ ]:

# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

# Construye un prompt que incluye el fragmento recuperado y la pregunta del usuario.
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

# Envía el prompt enriquecido con contexto al modelo.
Política: {fragmento}

# Muestra la respuesta final generada mediante el flujo RAG.
Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica

**Respuesta:** No. Solo están disponibles cambios de talla.
